In [0]:
dbutils.widgets.text("RUN_DATE", "")
RUN_DATE = dbutils.widgets.get("RUN_DATE").strip()

print("Incoming RUN_DATE =", RUN_DATE if RUN_DATE else "<empty>")


In [0]:
import pyspark.sql.functions as F

PIPELINE_NAME = "ecomm_events_daily_replay"
STATE_TABLE = "monitoring.pipeline_state"

def get_last_released_date():
    row = (
        spark.table(STATE_TABLE)
             .filter(F.col("pipeline_name") == PIPELINE_NAME)
             .select("last_released_date")
             .first()
    )
    if not row:
        raise ValueError("No pipeline_state row found")
    return str(row["last_released_date"])


In [0]:
if RUN_DATE:
    run_date = RUN_DATE
    print("Using manual RUN_DATE =", run_date)
else:
    print("No RUN_DATE provided → releasing next day from RAW")
    dbutils.notebook.run(
        "00_replay_to_landing",
        timeout_seconds=0,
        arguments={}
    )
    run_date = get_last_released_date()
    print("Released and using run_date =", run_date)


In [0]:
print("Running Bronze for", run_date)

dbutils.notebook.run(
    "01_bronze_ingestion",
    timeout_seconds=0,
    arguments={"RUN_DATE": run_date}
)


In [0]:
# After Bronze
print("Running Silver for", run_date)
dbutils.notebook.run(
    "03_silver_events_cleaning",
    timeout_seconds=0,
    arguments={"RUN_DATE": run_date}
)

print("Running Gold funnel metrics for", run_date)
dbutils.notebook.run(
    "04_gold_funnel_metrics",
    timeout_seconds=0,
    arguments={"RUN_DATE": run_date}
)

print("Running Gold product daily metrics for", run_date)
dbutils.notebook.run(
    "05_gold_product_daily_metrics",
    timeout_seconds=0,
    arguments={"RUN_DATE": run_date}
)
